In [ ]:
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import data_layer
from src.model.model_utils import calculateStatistics
from src.model.model_GAN import FeedForwardModelWithNA_GAN_Ensembled
from src.model.model_RtnFcst import FeedForwardModelWithNA_Return_Ensembled
from src.utils import deco_print
from src.utils import load_dataframe
from src.utils import sharpe

##### Model Configurations

In [ ]:
task_id = 1
N_TRIALS = 9
logdirs = [f"output/Task_1_Trial_{k}/sharpe" for k in range(N_TRIALS)]
with open("config/config.json") as file:
    config = json.load(file)
if "macro_idx" not in config:
    config["macro_idx"] = None

logdirs_RF = [f"output_RF/Task_1_Trial_{k}" for k in range(N_TRIALS)]
with open(f"config_RF/config_RF_{task_id}.json") as file:
    config_RF = json.load(file)

##### Load Data

In [3]:
dl = data_layer.DataInRamInputLayer(
    config["individual_feature_file"],
    pathMacroFeature=config["macro_feature_file"],
    macroIdx=config["macro_idx"],
)
meanMacroFeature, stdMacroFeature = dl.getMacroFeatureMeanStd()
dl_valid = data_layer.DataInRamInputLayer(
    config["individual_feature_file_valid"],
    pathMacroFeature=config["macro_feature_file_valid"],
    macroIdx=config["macro_idx"],
    meanMacroFeature=meanMacroFeature,
    stdMacroFeature=stdMacroFeature,
)
dl_test = data_layer.DataInRamInputLayer(
    config["individual_feature_file_test"],
    pathMacroFeature=config["macro_feature_file_test"],
    macroIdx=config["macro_idx"],
    meanMacroFeature=meanMacroFeature,
    stdMacroFeature=stdMacroFeature,
)

##### Load Model

In [4]:
model = FeedForwardModelWithNA_GAN_Ensembled(logdirs, config, "test")

In [ ]:
# Initial LSTM states are handled internally by the model; no manual threading needed.

##### Model Performance

In [ ]:
w       = model.getWeightWithData(dl,       normalized=True)
w_valid = model.getWeightWithData(dl_valid, normalized=True)
w_test  = model.getWeightWithData(dl_test,  normalized=True)

In [ ]:
Ftrain = model.getNormalizedSDF(dl)
Fvalid = model.getNormalizedSDF(dl_valid)
Ftest  = model.getNormalizedSDF(dl_test)
sdf_norm_ensemble = np.concatenate([Ftrain, Fvalid, Ftest])
os.makedirs("output", exist_ok=True)
np.save("output/sdf_normalized_ensemble.npy", sdf_norm_ensemble)

In [8]:
SR_train = sharpe(Ftrain)
SR_valid = sharpe(Fvalid)
SR_test = sharpe(Ftest)
deco_print('SDF Portfolio Sharpe Ratio: Train %0.2f\tValid %0.2f\tTest %0.2f' %(SR_train, SR_valid, SR_test))

##### Predictive Performance

In [9]:
model_RF = FeedForwardModelWithNA_Return_Ensembled(logdirs_RF, config_RF, "test")

In [10]:
dl_RF_train = data_layer.DataInRamInputLayer(config["individual_feature_file"])
dl_RF_valid = data_layer.DataInRamInputLayer(config["individual_feature_file_valid"])
dl_RF_test  = data_layer.DataInRamInputLayer(config["individual_feature_file_test"])

In [ ]:
beta_train = model_RF.getPrediction(dl_RF_train)
beta_valid = model_RF.getPrediction(dl_RF_valid)
beta_test  = model_RF.getPrediction(dl_RF_test)

In [12]:
# EV, XS-R2
EV_train, XSR2_train, WXSR2_train = calculateStatistics(beta_train, dl)
EV_valid, XSR2_valid, WXSR2_valid = calculateStatistics(beta_valid, dl_valid)
EV_test, XSR2_test, WXSR2_test = calculateStatistics(beta_test, dl_test)

In [13]:
deco_print('Explained Variation: Train %0.2f\tValid %0.2f\tTest %0.2f' %(EV_train, EV_valid, EV_test))
deco_print('XS-R2: Train %0.2f\tValid %0.2f\tTest %0.2f' %(XSR2_train, XSR2_valid, XSR2_test))
deco_print('(Weighted) XS-R2: Train %0.2f\tValid %0.2f\tTest %0.2f' %(WXSR2_train, WXSR2_valid, WXSR2_test))